# EEM-241 İleri Düzey Programlama

## 2024-2025 Güz Dönemi

### Ders 12 - OpenCV I: Görüntüyü Okuma ve Piksel İşlemleri

## 1. Kurulum ve çalışma ortamı

Gerekirse terminalde:

```text
pip install opencv-python
```


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


Bu derste `files/image.jpg` kullanılacaktır: [örnek görüntü](./files/image.jpg).

`cv2.imshow` masaüstü penceresi açabildiği ortamlarda çalışır. Jupyter içinde görüntüyü hücre altında göstermek için Matplotlib daha uygundur.

## 2. Görüntüyü güvenli biçimde okuma


In [ ]:
dosya_yolu = "files/image.jpg"
img = cv2.imread(dosya_yolu)

if img is None:
    raise FileNotFoundError(f"Görüntü okunamadı: {dosya_yolu}")


Dosya bulunamazsa `cv2.imread` istisna üretmek yerine `None` döndürür. Bu nedenle görüntü kullanılmadan önce kontrol edilmelidir.

Masaüstü penceresinde gösterme:


In [ ]:
cv2.imshow("Goruntu", img)
cv2.waitKey(0)
cv2.destroyAllWindows()


## 3. Görüntü bir NumPy dizisidir


In [ ]:
print(type(img))
print(img.shape)
print(img.ndim)
print(img.size)
print(img.dtype)


Renkli görüntünün şekli genellikle `(yükseklik, genişlik, kanal)` biçimindedir.


In [ ]:
yukseklik, genislik, kanal_sayisi = img.shape
print(yukseklik, genislik, kanal_sayisi)


Standart 8 bit görüntüde her kanal `0`–`255` arasında değer alır ve veri türü çoğunlukla `uint8`'dir.

## 4. BGR ve RGB farkı

OpenCV renkli görüntüyü BGR kanal sırasıyla okur; Matplotlib ise RGB bekler.


In [ ]:
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots()
ax.imshow(img_rgb)
ax.axis("off")
plt.show()


Dönüşüm yapılmadan Matplotlib ile gösterilen görüntüde kırmızı ve mavi kanallar yer değiştirmiş görünür.

## 5. Gri tonlamalı görüntü

Doğrudan gri okuma:


In [ ]:
img_gray = cv2.imread(dosya_yolu, cv2.IMREAD_GRAYSCALE)
print(img_gray.shape)


Renkli görüntüden dönüştürme:


In [ ]:
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

fig, ax = plt.subplots()
ax.imshow(img_gray, cmap="gray", vmin=0, vmax=255)
ax.axis("off")
plt.show()


Gri görüntü iki boyutlu, renkli görüntü üç boyutludur.

## 6. Koordinatlar ve piksel erişimi

OpenCV fonksiyonları çoğunlukla `(x, y)` yani `(sütun, satır)` kullanır. NumPy indekslemesi ise `[y, x]` yani `[satır, sütun]` biçimindedir.


In [ ]:
x, y = 116, 255
b, g, r = img[y, x]
gri_deger = img_gray[y, x]

print(b, g, r)
print(gri_deger)


Bir pikseli değiştirme:


In [ ]:
duzenlenmis = img.copy()
duzenlenmis[y, x] = (0, 0, 255)  # BGR: kırmızı


## 7. İlgi bölgesi (ROI)


In [ ]:
y1, y2 = 100, 270
x1, x2 = 120, 280
parca = img[y1:y2, x1:x2]

cv2.imshow("Parca", parca)
cv2.waitKey(0)
cv2.destroyAllWindows()


Dilim çoğunlukla orijinal görüntünün verisini paylaşır. Bağımsız parça gerektiğinde `.copy()` kullanılır:


In [ ]:
parca = img[y1:y2, x1:x2].copy()
parca[:, :, 1] = 255


### Bölgeyi başka konuma yapıştırma

Kaynak ve hedef bölgenin şekilleri aynı olmalıdır.


In [ ]:
sonuc = img.copy()
parca = sonuc[100:150, 120:170].copy()
sonuc[50:100, 300:350] = parca


### Dikdörtgen bir alanın rengini değiştirme


In [ ]:
sonuc = img.copy()
sonuc[50:120, 160:260] = (0, 255, 0)


## 8. Kanalları ayırma ve birleştirme


In [ ]:
b, g, r = cv2.split(img)
yeniden_birlestirilmis = cv2.merge((b, g, r))

print(b.shape, g.shape, r.shape)


Belirli bir renk kanalının etkisini görmek için diğer kanallar sıfırlanabilir:


In [ ]:
sifir = np.zeros_like(b)
yalnizca_kirmizi = cv2.merge((sifir, sifir, r))


## 9. Yeniden boyutlandırma

OpenCV'de hedef boyut `(genişlik, yükseklik)` sırasıyla verilir.


In [ ]:
kucuk = cv2.resize(img, (300, 150), interpolation=cv2.INTER_AREA)


En-boy oranını koruma:


In [ ]:
yeni_yukseklik = 200
oran = yeni_yukseklik / img.shape[0]
yeni_genislik = int(img.shape[1] * oran)

boyutlandirilmis = cv2.resize(
    img,
    (yeni_genislik, yeni_yukseklik),
    interpolation=cv2.INTER_AREA
)


Küçültmede `INTER_AREA`, büyütmede `INTER_LINEAR` makul başlangıç seçimleridir.

## 10. Görüntüyü kaydetme


In [ ]:
basarili = cv2.imwrite("files/sonuc.png", sonuc)

if not basarili:
    raise OSError("Görüntü dosyaya yazılamadı.")


`imwrite` başarı durumunu Boolean değer olarak döndürür.

## 11. Fare olayıyla piksel inceleme

Bu bölüm, temel görüntü işlemleri anlaşıldıktan sonra uygulanmalıdır.


In [ ]:
def mouse_event(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        b, g, r = img[y, x]
        print(f"x={x}, y={y}, BGR=({b}, {g}, {r})")


cv2.imshow("Goruntu", img)
cv2.setMouseCallback("Goruntu", mouse_event)
cv2.waitKey(0)
cv2.destroyAllWindows()


## 12. Bütünleştirici uygulama


In [ ]:
img = cv2.imread("files/image.jpg")
if img is None:
    raise FileNotFoundError("Görüntü bulunamadı.")

gri = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
yukseklik, genislik = gri.shape

merkez = gri[yukseklik // 4:3 * yukseklik // 4,
             genislik // 4:3 * genislik // 4].copy()

merkez = cv2.resize(merkez, (300, 200), interpolation=cv2.INTER_AREA)
cv2.imwrite("files/merkez_gri.png", merkez)


## Ders sonu alıştırmaları

1. Bir görüntünün boyutunu, kanal sayısını, veri türünü ve bellek kullanımını yazdırın.
2. Görüntünün sol yarısını gri tonlamalı yapıp sağ yarısını renkli bırakın.
3. Görüntünün merkezindeki `100 × 100` piksellik alanı çıkarıp ayrı dosyaya kaydedin.
4. Görüntüyü en-boy oranını koruyarak genişliği 640 piksel olacak şekilde boyutlandırın.
5. Fareyle tıklanan pikselin koordinatını ve BGR değerini görüntünün üzerine yazdırın.
